In [10]:
import pandas as pd
import numpy as np
import os

# Load merged dataset
merged_path = "../data/interim/asrs_merged_3yrs.csv"
df = pd.read_csv(merged_path)

df.shape
df.head()


C:\Users\jenny\AppData\Local\Temp\ipykernel_3732\2804889118.py:7: DtypeWarning: Columns (7,8,15,19,20,41,48,59,63,78,79,83,89,100,110,111,123) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(merged_path)


,ACN,Date,Local Time Of Day,Locale Reference,State Reference,Relative Position.Angle.Radial,Relative Position.Distance.Nautical Miles,Altitude.AGL.Single Value,Altitude.MSL.Single Value,Latitude / Longitude (UAS),...,When Detected,Result,Contributing Factors / Situations,Primary Problem,Narrative,Callback,Narrative.1,Callback.1,Synopsis,source_file
0,1714400,202001.0,0001-0600,ORD.Airport,IL,NaN,NaN,NaN,15000.0,NaN,...,In-flight,Air Traffic Control Issued New Clearance; Air ...,Aircraft; Company Policy,Company Policy,To be clear I did not have a lot going on and ...,NaN,NaN,NaN,A TRACON Departure Controller reported two air...,ASRS_DBOnline_012020-062020.csv
1,1714553,202001.0,1801-2400,ZHU.ARTCC,TX,NaN,NaN,NaN,14000.0,NaN,...,In-flight,Flight Crew Became Reoriented; General Physica...,Human Factors; Environment - Non Weather Related,Environment - Non Weather Related,While descending into PIB out of 14;000 ft. MS...,NaN,NaN,NaN,CRJ Captain reported that a laser shone repeat...,ASRS_DBOnline_012020-062020.csv
2,1714675,202001.0,1801-2400,ZZZ.ARTCC,US,NaN,NaN,NaN,9000.0,NaN,...,In-flight,Flight Crew Returned To Departure Airport; Fli...,Aircraft; Weather; Procedure,Weather,I was boarding the aircraft when a my destinat...,NaN,NaN,NaN,Pilot reported weather and a closed destinatio...,ASRS_DBOnline_012020-062020.csv
3,1714708,202001.0,0601-1200,ZZZ.Airport,US,NaN,NaN,0.0,NaN,NaN,...,In-flight,General None Reported / Taken,Procedure; Logbook Entry; Human Factors,Ambiguous,I had a ferry flight that went out of service ...,NaN,We departed ZZZ ferrying an un-airworthy aircr...,NaN,Air carrier Dispatcher and flight crew reporte...,ASRS_DBOnline_012020-062020.csv
4,1714843,202001.0,1801-2400,OSU.Airport,OH,NaN,0.0,0.0,NaN,NaN,...,In-flight,Flight Crew Diverted; Flight Crew Landed in Em...,Aircraft; Airport,Aircraft,We were flying at 2500 ft. I started losing el...,NaN,NaN,NaN,A Pilot reported a loss of electrical power at...,ASRS_DBOnline_012020-062020.csv


In [11]:
[n for n in df.columns if "Report" in n or "Event" in n or "Narr" in n]

['Reporter Organization',
 'ASRS Report Number.Accession Number',
 'Reporter Organization.1',
 'ASRS Report Number.Accession Number.1',
 'Were Passengers Involved In Event',
 'Narrative',
 'Narrative.1']

In [12]:
def clean_text(s):
    if pd.isna(s):
        return s
    s = str(s)
    s = s.replace("\n", " ").replace("\r", " ")
    s = s.replace("\t", " ")
    s = s.encode("utf-8", "ignore").decode("utf-8", "ignore")
    s = " ".join(s.split())  # collapse multiple spaces
    return s

narrative_cols = [c for c in df.columns if "Report" in c or "Event" in c]

for col in narrative_cols:
    df[col] = df[col].apply(clean_text)

df[narrative_cols].head()

,Reporter Organization,ASRS Report Number.Accession Number,Reporter Organization.1,ASRS Report Number.Accession Number.1,Were Passengers Involved In Event
0,Government,1714400,NaN,NaN,NaN
1,Air Carrier,1714553,NaN,NaN,NaN
2,Air Carrier,1714675,NaN,NaN,NaN
3,Air Carrier,1714708,Air Carrier,1715032.0,NaN
4,Personal,1714843,NaN,NaN,NaN


In [13]:
[c for c in df.columns if "Time" in c or "Date" in c]


['Date', 'Local Time Of Day']

In [14]:
# normalize date field
df['Date'] = df['Date'].astype(str).str.strip()

# Convert YYYYMM → YYYY-MM
df['Date'] = df['Date'].str[:4] + "-" + df['Date'].str[4:6]

df['Date'].head()

0    2020-01
1    2020-01
2    2020-01
3    2020-01
4    2020-01
Name: Date, dtype: object

In [15]:
def time_bucket(t):
    if pd.isna(t):
        return np.nan
    t = str(t)
    if "0001" in t: return "Night"
    if "0601" in t: return "Morning"
    if "1201" in t: return "Afternoon"
    if "1801" in t: return "Evening"
    return "Unknown"

df['TimeOfDayBucket'] = df['Local Time Of Day'].apply(time_bucket)
df[['Local Time Of Day', 'TimeOfDayBucket']].head()

,Local Time Of Day,TimeOfDayBucket
0,0001-0600,Night
1,1801-2400,Evening
2,1801-2400,Evening
3,0601-1200,Morning
4,1801-2400,Evening


In [17]:
df['Locale Reference'] = df['Locale Reference'].astype(str).str.strip()
df['State Reference'] = df['State Reference'].astype(str).str.strip()

# Optional: clean latitude/longitude if needed
if 'Latitude / Longitude (UAS)' in df.columns:
    df['Latitude / Longitude (UAS)'] = df['Latitude / Longitude (UAS)'].astype(str).str.strip()

In [18]:
processed_path = "../data/processed/"
os.makedirs(processed_path, exist_ok=True)

output_file = os.path.join(processed_path, "asrs_cleaned_3yrs.csv")
df.to_csv(output_file, index=False)

print("Saved cleaned dataset to:", output_file)
print("Final shape:", df.shape)


Saved cleaned dataset to: ../data/processed/asrs_cleaned_3yrs.csv
Final shape: (16535, 127)
